# Matrixsteifigkeitsmethode: 2D-Fachwerk

In diesem Notebook baust du dir einen kleinen Fachwerk-Löser. Nicht weil du später alles von Hand programmieren musst, sondern weil man dabei sehr gut sieht, **was die Methode wirklich macht**.

Heute schauen wir nur auf:
- **Verschiebungen** $U$
- **Kräfte** $F$ (inkl. Reaktionen an den gelagerten Freiheitsgraden)

Dehnungen und Spannungen lassen wir bewusst weg. Die kommen erst, wenn die Grundmaschine sauber läuft.


## Wie du hier vorgehst

In den Zellen findest du Platzhalter wie `# TODO: ...` oder `...` (drei Punkte). Du sollst jeweils **nur den Ausdruck rechts** einsetzen.

Wir gehen von klein nach gross:
1. **Elementebene:**
    - $\underline{\underline{k}}_{\text{lokal}}$: lokale Elementsteifigkeitsmatrix
    - $\underline{\underline{T}}$: Transformationsmatrix
    - $\underline{\underline{K}}_e$: globale Elementsteifigkeitsmatrix
2. **Strukturebene:**
    - Koinzidenztabelle (engl. incidence table)
    - $\underline{\underline{K}}$: aus Elementsteifigkeitsmatrizen assemblierte globale Struktursteifigkeitsmatrix (SSM)
3. **Randbedingungen:**
    - eine Zeile für $\boldsymbol{U}_F$

## Imports

Wir brauchen nur NumPy.


In [ ]:
import numpy as np


## Modell

Das ist unser Beispiel-Fachwerk. Die Daten sind fertig, damit du dich auf die Methode konzentrieren kannst.


In [ ]:
# Knotenkoordinaten [X, Y] in mm
nodal_coordinates = np.array([
    [   0.0,    0.0],
    [1000.0,    0.0],
    [1000.0, 1000.0],
    [2000.0, 1000.0],
])

# Elemente: [Knoten_i, Knoten_j, section_key]
elements = [
    [0, 1, "section 1"],
    [0, 2, "section 2"],
    [1, 2, "section 3"],
    [1, 3, "section 4"],
    [2, 3, "section 5"],
]

# Materialien: Youngscher Modul [MPa = N/mm²]
materials = {"steel": [210000.0]}

# Querschnitte: [A [mm²], material_key]
sections = {
    "section 1": [15.00, "steel"],
    "section 2": [28.28, "steel"],
    "section 3": [10.00, "steel"],
    "section 4": [56.56, "steel"],
    "section 5": [10.00, "steel"],
}

# Randbedingungen: [Knoten (0-basiert), Achse (0=x,1=y), vorgegebene Verschiebung]
constraints = [
    [0, 0, 0.0],   # Knoten 1: U_1 = 0 (Festlager, x)
    [0, 1, 0.0],   # Knoten 1: U_2 = 0 (Festlager, y)
    [1, 1, 0.0],   # Knoten 2: U_4 = 0 (Loslager, y)
]

# Lasten: [Knoten (0-basiert), Achse (0=x,1=y), Kraft [N]]
loads = [
    [3, 1, -1000.0],   # Knoten 4: F_y = -1000 N
]

## 1) Elementebene: $\underline{\underline{k}}_{\text{lokal}}$, $\underline{\underline{T}}$, $\underline{\underline{K}}_e$

Hier ist alles als Code-Gerüst vorhanden. Du füllst nur die Ausdrücke bei den `...`.

Formeln aus der Vorlesung (in der Reihenfolge aus dem Code):

- $dx = x_j - x_i$, $\;dy = y_j - y_i$: Koordinatendifferenzen des Elements.
- $L = \sqrt{dx^2 + dy^2}$: Elementlänge.
- $c = \cos\theta = dx/L$, $\;s = \sin\theta = dy/L$: Richtungskosinusse.
- $\underline{\underline{k}}_{\text{lokal}} = \dfrac{EA}{L}\begin{bmatrix}1&-1\\-1&1\end{bmatrix}$ $\;\longrightarrow\;$ `k_lokal = (EA/L) * np.array([[1,-1],[-1,1]])`
- $\underline{\underline{T}} = \begin{bmatrix}c & s & 0 & 0\\ 0 & 0 & c & s\end{bmatrix}$ $\;\longrightarrow\;$ `T = np.array([[c, s, 0, 0], [0, 0, c, s]])`
- $\underline{\underline{K}}_e = \underline{\underline{T}}^T\,\underline{\underline{k}}_{\text{lokal}}\,\underline{\underline{T}}$ $\;\longrightarrow\;$ `Ke = T.T @ k_lokal @ T`

In [ ]:
def element_stiffness_matrix(EA, xy_e):
    """Globale Elementsteifigkeitsmatrix K_e für ein 2D-Stab-Element."""
    dx = xy_e[1, 0] - xy_e[0, 0]
    dy = xy_e[1, 1] - xy_e[0, 1]
    L  = np.sqrt(dx**2 + dy**2)

    c = dx / L   # cos(theta)
    s = dy / L   # sin(theta)

    # TODO: lokale Elementsteifigkeit (2×2)
    k_lokal = ...

    # TODO: Transformationsmatrix (2×4)
    T = ...

    # TODO: globale Elementsteifigkeitsmatrix (4×4)
    Ke = ...

    return Ke

**Hinweis:** Die globale Elementsteifigkeitsmatrix $\underline{\underline{K}}_e = \underline{\underline{T}}^T\,\underline{\underline{k}}_{\text{lokal}}\,\underline{\underline{T}}$ lässt sich auch direkt ausrechnen:

$$\underline{\underline{K}}_e = \frac{EA}{L}\begin{bmatrix}c^2 & cs & -c^2 & -cs\\ cs & s^2 & -cs & -s^2\\ -c^2 & -cs & c^2 & cs\\ -cs & -s^2 & cs & s^2\end{bmatrix}, \qquad c = \cos\theta,\quad s = \sin\theta$$

## 2) Koinzidenztabelle (engl. incidence table)

Die **Koinzidenztabelle** beschreibt, an welchen Positionen in der globalen SSM $\underline{\underline{K}}$ die Einträge der jeweiligen $\underline{\underline{K}}_e$ abgelegt werden:
- Jeder Knoten hat zwei Freiheitsgrade: $u_x = 2i$, $u_y = 2i+1$.
- Für jedes Element entsteht ein Vektor $[2i,\; 2i+1,\; 2j,\; 2j+1]$.

In [ ]:
def incidence_table(elements):
    """Koinzidenztabelle (engl. incidence table):
    Gibt die globalen DOF-Indizes (0-basiert) je Element zurück."""
    conn = np.array([[e[0], e[1]] for e in elements], dtype=int)
    dofs = np.vstack((
        2 * conn[:, 0],
        2 * conn[:, 0] + 1,
        2 * conn[:, 1],
        2 * conn[:, 1] + 1,
    )).T
    return dofs

## 3) Assemblierung: globale SSM $\underline{\underline{K}}$

Diese Funktion ist fertig. Sie nimmt `incidence_table` und `element_stiffness_matrix` und assembliert daraus $\underline{\underline{K}}$:

$$\underline{\underline{K}}[\text{DOFs}^{(e)},\,\text{DOFs}^{(e)}] \mathrel{+}= \underline{\underline{K}}_e$$

In [ ]:
def assemble_K(nodal_coordinates, elements, sections, materials):
    dofs = incidence_table(elements)
    ndof = int(np.max(dofs) + 1)
    K = np.zeros((ndof, ndof))

    for e, (i, j, sec_key) in enumerate(elements):
        A, mat_key = sections[sec_key]
        E = materials[mat_key][0]
        EA = E * A

        xy_e = nodal_coordinates[[i, j], :]
        Ke = element_stiffness_matrix(EA, xy_e)

        edofs = dofs[e]
        K[np.ix_(edofs, edofs)] += Ke

    return K


## 4) Randbedingungen und Lösen

Auch hier steht fast alles. Du füllst nur **eine** Zeile:

$$
\boldsymbol{U}_F = \underline{\underline{K}}_{FF}^{-1}\,(\boldsymbol{F}_F - \underline{\underline{K}}_{FU}\,\boldsymbol{U}_U)
$$

In NumPy: `np.linalg.solve(K_FF, rhs)` löst $\underline{\underline{K}}_{FF}\,x = rhs$.

In [ ]:
def solve_system(K, constraints, loads):
    ndof = K.shape[0]

    _U = np.zeros(ndof, dtype=bool)
    U_U = []

    for node, axis, val in constraints:
        dof = 2*int(node) + int(axis)
        _U[dof] = True
        U_U.append(val)

    U_U = np.array(U_U, dtype=float)
    _F = ~_U

    F = np.zeros(ndof)
    for node, axis, val in loads:
        dof = 2*int(node) + int(axis)
        F[dof] = val

    K_FF = K[np.ix_(_F, _F)]
    K_FU = K[np.ix_(_F, _U)]
    F_F  = F[_F]

    # TODO: freie Verschiebungen
    U_F = ...

    U = np.zeros(ndof)
    U[_U] = U_U
    U[_F] = U_F

    # Reaktionen
    K_UF = K[np.ix_(_U, _F)]
    K_UU = K[np.ix_(_U, _U)]
    F_U = K_UF @ U_F + K_UU @ U_U
    F[_U] = F_U

    return U, F, _U


## Ausführen

Wenn du die `...` ersetzt hast, kannst du rechnen.

Mini-Check für dich:
- `K` muss symmetrisch sein.
- an festen DOFs sind Verschiebungen 0.


In [ ]:
K = assemble_K(nodal_coordinates, elements, sections, materials)
U, F, fixed_mask = solve_system(K, constraints, loads)

np.set_printoptions(precision=6, suppress=True)
print("Verschiebungen U [mm]:")
print(U)

print("\nKräfte F [N] (inkl. Reaktionen an festen DOFs):")
print(F)

print("\nSymmetrie-Check K  max|K − Kᵀ| =", np.max(np.abs(K - K.T)))